# 📈 Time Series Forecasting for Business Operations

## Project Goal

In this project, we will predict **future business sales** using historical daily sales data.

We will learn the complete workflow:

1. Load and understand data
2. Clean the data
3. Perform Exploratory Data Analysis (EDA)
4. Convert daily sales into monthly sales
5. Check trend and seasonality
6. Split data into train and test
7. Build a **Baseline model**
8. Build an **ARIMA/SARIMA model**
9. Build an **LSTM Deep Learning model**
10. Compare models using MAE, RMSE and MAPE
11. Forecast future sales
12. Save the final forecast
13. Understand how to explain this project in an interview

> **Fresher Note:** Time series means data collected over time, for example sales every day, website visitors every hour, or electricity usage every month.

## 🧰 Libraries Used

- **Pandas** → data handling
- **NumPy** → numerical calculations
- **Matplotlib** → charts
- **Seaborn** → visualization
- **Statsmodels** → time-series analysis and SARIMA
- **Scikit-learn** → scaling and evaluation metrics
- **TensorFlow/Keras** → LSTM deep learning model

### Install libraries

Run this in your terminal/Jupyter if needed:

```bash
pip install pandas numpy matplotlib seaborn statsmodels scikit-learn tensorflow openpyxl
```

In [1]:
# Import libraries

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'statsmodels'

# 1. Create a Business Sales Dataset

For learning, we will create a realistic-looking business sales dataset.

The dataset contains:

- Date
- Sales

The data will contain:
- long-term trend
- monthly/weekly patterns
- random noise

> **Real Project:** Later, replace this generated data with your company's or Kaggle dataset.

In [ ]:
# Create sample daily sales data

np.random.seed(42)

dates = pd.date_range(
    start="2021-01-01",
    end="2025-12-31",
    freq="D"
)

n = len(dates)

# Long-term upward trend
trend = np.linspace(100, 220, n)

# Weekly seasonality
weekly_seasonality = 20 * np.sin(2 * np.pi * np.arange(n) / 7)

# Yearly seasonality
yearly_seasonality = 35 * np.sin(2 * np.pi * np.arange(n) / 365.25)

# Random noise
noise = np.random.normal(0, 12, n)

sales = trend + weekly_seasonality + yearly_seasonality + noise

# Make sales positive
sales = np.maximum(sales, 20)

df = pd.DataFrame({
    "Date": dates,
    "Sales": sales.round(2)
})

df.head()

## If you have your own CSV file

Use this instead:

```python
df = pd.read_csv("sales.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date")

df = df[["Date", "Sales"]]
```

Your CSV should look like:

| Date | Sales |
|---|---:|
| 2021-01-01 | 1200 |
| 2021-01-02 | 1350 |
| 2021-01-03 | 1280 |

> **Important:** The date column must contain dates and the target column should contain numerical sales.

# 2. Understand the Dataset

Before building a model, always understand the data.

This is a very important habit for a Data Analyst/Data Scientist.

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nSummary:")
display(df.describe())

In [ ]:
# Check first and last records

display(df.head())
display(df.tail())

# 3. Data Cleaning

Common time-series cleaning tasks:

- Convert date into datetime
- Sort dates
- Remove duplicates
- Handle missing values
- Set date as index

The **index** is important because time-series libraries work better with dates as the index.

In [ ]:
# Convert Date to datetime

df["Date"] = pd.to_datetime(df["Date"])

# Sort by date
df = df.sort_values("Date")

# Remove duplicate dates
df = df.drop_duplicates(subset="Date")

# Set Date as index
df = df.set_index("Date")

# Make sure Sales is numeric
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")

# Handle missing sales values
df["Sales"] = df["Sales"].interpolate()

print(df.head())
print("\nMissing values:", df.isnull().sum().sum())

# 4. Plot Daily Sales

A graph helps us visually understand:

- Trend → Is sales increasing or decreasing?
- Seasonality → Does a pattern repeat?
- Anomalies → Are there unusual spikes or drops?

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(df.index, df["Sales"])

plt.title("Daily Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Sales")

plt.show()

# 5. Convert Daily Sales to Monthly Sales

Business forecasting is often easier at a monthly level.

`resample("MS")` means:

> Group the data month by month, starting at the beginning of each month.

In [ ]:
monthly_sales = df["Sales"].resample("MS").sum()

monthly_sales.head()

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(monthly_sales.index, monthly_sales)

plt.title("Monthly Sales")
plt.xlabel("Date")
plt.ylabel("Total Sales")

plt.show()

# 6. Explore Trend and Seasonality

A time series can contain:

### Trend
Long-term movement.

Example:
Sales are increasing every year.

### Seasonality
A pattern that repeats at a fixed interval.

Example:
Sales increase every December.

### Residual
Random/unexplained movement after trend and seasonality are removed.

In [ ]:
# Seasonal decomposition

decomposition = seasonal_decompose(
    monthly_sales,
    model="additive",
    period=12
)

decomposition.plot(figsize=(15, 10))

plt.show()

# 7. Monthly Seasonality

Let's calculate the average sales for each month.

This helps answer:

> Which months normally have higher sales?

In [ ]:
monthly_pattern = monthly_sales.groupby(monthly_sales.index.month).mean()

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

monthly_pattern.index = month_names

plt.figure(figsize=(12, 5))

sns.barplot(
    x=monthly_pattern.index,
    y=monthly_pattern.values
)

plt.title("Average Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Sales")

plt.show()

# 8. Create Lag Features

A lag means previous value.

Example:

- Lag 1 → previous month
- Lag 2 → two months ago
- Lag 12 → same month last year

Lag features are very useful in time-series machine learning.

In [ ]:
feature_df = pd.DataFrame(index=monthly_sales.index)

feature_df["Sales"] = monthly_sales

feature_df["Lag_1"] = feature_df["Sales"].shift(1)
feature_df["Lag_3"] = feature_df["Sales"].shift(3)
feature_df["Lag_12"] = feature_df["Sales"].shift(12)

feature_df["Rolling_3"] = feature_df["Sales"].rolling(3).mean()
feature_df["Rolling_12"] = feature_df["Sales"].rolling(12).mean()

display(feature_df.tail())

# 9. Train-Test Split

We must **not randomly shuffle** time-series data.

Why?

Because the future should never be used to predict the past.

We will use:

- **80% → Training data**
- **20% → Testing data**

The test data represents unseen future data.

In [ ]:
split_index = int(len(monthly_sales) * 0.80)

train = monthly_sales.iloc[:split_index]
test = monthly_sales.iloc[split_index:]

print("Training observations:", len(train))
print("Testing observations:", len(test))

print("Train end:", train.index[-1])
print("Test start:", test.index[0])

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(train.index, train, label="Train")
plt.plot(test.index, test, label="Test")

plt.title("Train-Test Split")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

# 10. Baseline Model

Before using complicated models, create a simple baseline.

Here we predict:

> Next month's sales = previous month's sales

This gives us a benchmark.

If our advanced model cannot beat the baseline, something may be wrong.

In [ ]:
# Naive forecast

naive_predictions = test.shift(1)

# First test prediction uses the last training value
naive_predictions.iloc[0] = train.iloc[-1]

naive_predictions.index = test.index

naive_mae = mean_absolute_error(test, naive_predictions)

naive_rmse = np.sqrt(
    mean_squared_error(test, naive_predictions)
)

naive_mape = np.mean(
    np.abs((test - naive_predictions) / test)
) * 100

print("Baseline MAE:", round(naive_mae, 2))
print("Baseline RMSE:", round(naive_rmse, 2))
print("Baseline MAPE:", round(naive_mape, 2), "%")

# 11. ARIMA / SARIMA Model

## ARIMA

ARIMA stands for:

- **AR** → Auto Regression
- **I** → Integrated / Differencing
- **MA** → Moving Average

ARIMA is useful for time-series forecasting.

## SARIMA

SARIMA adds seasonality to ARIMA.

Our data is monthly, so a yearly seasonal pattern has a period of **12 months**.

We will use:

```text
SARIMA(1,1,1)(1,1,1,12)
```

Don't worry about memorizing these numbers initially. Understand what the model is doing first.

In [ ]:
# Build SARIMA model

sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_result = sarima_model.fit(disp=False)

print(sarima_result.summary())

In [ ]:
# Forecast test period

sarima_forecast = sarima_result.get_forecast(
    steps=len(test)
)

sarima_predictions = sarima_forecast.predicted_mean

sarima_mae = mean_absolute_error(
    test,
    sarima_predictions
)

sarima_rmse = np.sqrt(
    mean_squared_error(
        test,
        sarima_predictions
    )
)

sarima_mape = np.mean(
    np.abs((test - sarima_predictions) / test)
) * 100

print("SARIMA MAE:", round(sarima_mae, 2))
print("SARIMA RMSE:", round(sarima_rmse, 2))
print("SARIMA MAPE:", round(sarima_mape, 2), "%")

In [ ]:
# Plot SARIMA forecast

plt.figure(figsize=(15, 6))

plt.plot(train.index, train, label="Train")
plt.plot(test.index, test, label="Actual")
plt.plot(
    test.index,
    sarima_predictions,
    label="SARIMA Forecast"
)

plt.title("SARIMA Forecast vs Actual")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

# 12. LSTM Deep Learning Model

LSTM = **Long Short-Term Memory**.

LSTM is a type of neural network designed to learn patterns from sequential data.

Simple idea:

> LSTM looks at previous sales values and learns how they are related to future sales.

For LSTM, we need to convert the time series into sequences.

Example with `lookback = 12`:

```text
Previous 12 months → Next month
```

So the model learns from the previous 12 months to predict the next month.

In [ ]:
# Import TensorFlow

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

print("TensorFlow version:", tf.__version__)

## Scale the Data

Neural networks usually work better when values are on a smaller scale.

We will convert sales approximately into a range between 0 and 1 using `MinMaxScaler`.

Important:

> Fit the scaler only on training data to avoid data leakage.

In [ ]:
# Scale data

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(
    train.values.reshape(-1, 1)
)

test_scaled = scaler.transform(
    test.values.reshape(-1, 1)
)

print("Scaled training data shape:", train_scaled.shape)

# 13. Create LSTM Sequences

We use 12 previous months to predict the next month.

Example:

```text
Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec → Jan
Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec Jan → Feb
```

This is called a **sliding window**.

In [ ]:
lookback = 12

def create_sequences(data, lookback):
    X = []
    y = []

    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i, 0])
        y.append(data[i, 0])

    return np.array(X), np.array(y)

X_train, y_train = create_sequences(
    train_scaled,
    lookback
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

# 14. Prepare Test Data for LSTM

The first test prediction needs the last 12 months from training data.

So we combine:

```text
last 12 training values + test values
```

Then create test sequences.

In [ ]:
combined_scaled = np.concatenate(
    [train_scaled[-lookback:], test_scaled]
)

X_test, y_test = create_sequences(
    combined_scaled,
    lookback
)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# 15. Reshape Data for LSTM

LSTM expects 3 dimensions:

```text
(samples, time_steps, features)
```

For our project:

```text
samples = number of sequences
time_steps = 12 months
features = 1 sales value
```

In [ ]:
X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

print("Final X_train shape:", X_train.shape)
print("Final X_test shape:", X_test.shape)

# 16. Build the LSTM Model

Architecture:

```text
Input
 ↓
LSTM
 ↓
Dropout
 ↓
LSTM
 ↓
Dense
 ↓
Forecast
```

### Dropout

Dropout randomly ignores some neurons during training.

This can help reduce overfitting.

In [ ]:
# Build LSTM model

lstm_model = Sequential([
    LSTM(
        64,
        return_sequences=True,
        input_shape=(lookback, 1)
    ),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

lstm_model.summary()

# 17. Train the LSTM Model

We use:

- `epochs=100` → maximum training rounds
- `batch_size=16` → number of samples processed together
- `validation_split=0.1` → 10% of training data used for validation

`EarlyStopping` stops training when validation loss stops improving.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Plot training and validation loss

plt.figure(figsize=(12, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

# 18. Predict with LSTM

The model gives predictions in the scaled 0–1 range.

We must convert them back to the original sales scale using `inverse_transform()`.

In [ ]:
# Predict

lstm_scaled_predictions = lstm_model.predict(
    X_test
)

# Convert predictions back to original scale
lstm_predictions = scaler.inverse_transform(
    lstm_scaled_predictions
).flatten()

# Actual values
actual_lstm = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).flatten()

lstm_mae = mean_absolute_error(
    actual_lstm,
    lstm_predictions
)

lstm_rmse = np.sqrt(
    mean_squared_error(
        actual_lstm,
        lstm_predictions
    )
)

lstm_mape = np.mean(
    np.abs(
        (actual_lstm - lstm_predictions)
        / actual_lstm
    )
) * 100

print("LSTM MAE:", round(lstm_mae, 2))
print("LSTM RMSE:", round(lstm_rmse, 2))
print("LSTM MAPE:", round(lstm_mape, 2), "%")

In [ ]:
# Create LSTM prediction series

lstm_predictions_series = pd.Series(
    lstm_predictions,
    index=test.index
)

plt.figure(figsize=(15, 6))

plt.plot(test.index, test, label="Actual")
plt.plot(
    test.index,
    lstm_predictions_series,
    label="LSTM Forecast"
)

plt.title("LSTM Forecast vs Actual")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

# 19. Compare All Models

We will compare:

1. Baseline
2. SARIMA
3. LSTM

### Metrics

**MAE — Mean Absolute Error**

Average absolute prediction error.

Lower = better.

**RMSE — Root Mean Squared Error**

Penalizes large errors more strongly.

Lower = better.

**MAPE — Mean Absolute Percentage Error**

Shows error as a percentage.

Lower = better.

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline", "SARIMA", "LSTM"],
    "MAE": [
        naive_mae,
        sarima_mae,
        lstm_mae
    ],
    "RMSE": [
        naive_rmse,
        sarima_rmse,
        lstm_rmse
    ],
    "MAPE": [
        naive_mape,
        sarima_mape,
        lstm_mape
    ]
})

results = results.sort_values("MAE")

display(results.round(2))

# 20. Visual Model Comparison

A visual comparison makes it easier to understand which model follows actual sales more closely.

In [ ]:
plt.figure(figsize=(15, 7))

plt.plot(
    test.index,
    test,
    label="Actual"
)

plt.plot(
    test.index,
    sarima_predictions,
    label="SARIMA"
)

plt.plot(
    test.index,
    lstm_predictions_series,
    label="LSTM"
)

plt.title("Actual vs Forecasts")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

# 21. Select the Best Model

We select the model with the lowest MAE.

In a real company, don't select a model only because it is more advanced.

A simple model can sometimes perform better than deep learning.

Also consider:

- accuracy
- training time
- explainability
- maintenance
- business requirements

In [ ]:
best_model = results.iloc[0]

print("Best model:", best_model["Model"])
print("MAE:", round(best_model["MAE"], 2))
print("RMSE:", round(best_model["RMSE"], 2))
print("MAPE:", round(best_model["MAPE"], 2), "%")

# 22. Forecast Future Sales

Now we train the SARIMA model on the complete historical dataset and forecast the next 12 months.

For a real project, you can choose the best model based on validation performance.

Here we show a 12-month SARIMA forecast.

In [ ]:
# Train SARIMA on all available monthly data

final_sarima = SARIMAX(
    monthly_sales,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_sarima_result = final_sarima.fit(disp=False)

# Forecast next 12 months
future_steps = 12

future_forecast = final_sarima_result.get_forecast(
    steps=future_steps
)

future_sales = future_forecast.predicted_mean

print("Next 12 months forecast:")
display(future_sales.to_frame("Forecast_Sales").round(2))

In [ ]:
# Plot future forecast

plt.figure(figsize=(15, 6))

plt.plot(
    monthly_sales.index,
    monthly_sales,
    label="Historical Sales"
)

plt.plot(
    future_sales.index,
    future_sales,
    label="Future Forecast"
)

plt.title("Next 12 Months Sales Forecast")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

# 23. Add Forecast Confidence Interval

A forecast is not always one exact number.

We can also calculate a range where the future value may fall.

This is called a **confidence interval**.

For business users, this is useful for planning.

In [ ]:
forecast_frame = future_forecast.summary_frame()

forecast_frame = forecast_frame[
    [
        "mean",
        "mean_ci_lower",
        "mean_ci_upper"
    ]
]

forecast_frame.columns = [
    "Forecast",
    "Lower_Bound",
    "Upper_Bound"
]

display(forecast_frame.round(2))

In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(
    monthly_sales.index,
    monthly_sales,
    label="Historical"
)

plt.plot(
    forecast_frame.index,
    forecast_frame["Forecast"],
    label="Forecast"
)

plt.fill_between(
    forecast_frame.index,
    forecast_frame["Lower_Bound"],
    forecast_frame["Upper_Bound"],
    alpha=0.2,
    label="Confidence Interval"
)

plt.title("Sales Forecast with Confidence Interval")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

# 24. Save Forecast to CSV

A business team may want the forecast as an Excel/CSV file.

We can export it.

In [ ]:
# Save forecast

forecast_output = forecast_frame.reset_index()

forecast_output.rename(
    columns={"index": "Date"},
    inplace=True
)

forecast_output.to_csv(
    "sales_forecast_12_months.csv",
    index=False
)

print("Forecast saved as sales_forecast_12_months.csv")

display(forecast_output.round(2))

# 25. Business Insights

After forecasting, don't stop at the model.

A Data Analyst should convert numbers into business insights.

For example:

### Sales Planning
The company can estimate expected future sales.

### Inventory Management
If sales are expected to increase, inventory can be increased before demand arrives.

### Staffing
More expected sales may require more employees.

### Marketing
Marketing campaigns can be planned around seasonal demand.

### Budgeting
Finance teams can use forecasts for revenue planning.

> **Important:** A good data professional explains both the model and the business meaning.

In [ ]:
# Simple business summary

forecast_avg = forecast_frame["Forecast"].mean()
last_actual = monthly_sales.iloc[-1]

growth = (
    (forecast_avg - last_actual)
    / last_actual
) * 100

print("Last actual monthly sales:", round(last_actual, 2))
print("Average forecasted monthly sales:", round(forecast_avg, 2))
print("Approx. change from last month:", round(growth, 2), "%")